# Competencies pipeline

Цель: извлечь компетенции из текста вакансий, собрать one-hot признаки, отфильтровать топ‑30 компетенций и сохранить датасеты для обучения.

**Вход:** `data/vacancies_labeled_ml_dl.csv`

**Выход:**
- `vacancies_with_competencies.csv`
- `vacancies_top30_competencies.csv`
- `vacancies_top30_competencies_train_ready.csv`

In [10]:
from __future__ import annotations

from collections import Counter
import re

import numpy as np
import pandas as pd

In [12]:
# =========================
# Config
# =========================
from pathlib import Path

# Notebook is in `<repo>/notebooks/`, so project root is one level up.
ROOT_DIR = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
DATA_DIR = ROOT_DIR / "data"
ARTIFACTS_DIR = ROOT_DIR / "artifacts"
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

INPUT_FILE = DATA_DIR / "vacancies_labeled_ml_dl.csv"
OUT_TRAIN_READY = ARTIFACTS_DIR / "vacancies_train_ready.csv"

TEXT_COLUMNS_CANDIDATES = ["description", "vacancy_text", "text", "body"]
TOP_N = 30

# Currency rates -> RUB (adjust if needed)
CURRENCY_RATES_TO_RUB = {
    "RUB": 1.0,
    "RUR": 1.0,
    "USD": 92.0,
    "EUR": 100.0,
    "KZT": 0.20,
    "BYN": 28.0,
    "UAH": 2.2,
    "UZS": 0.0073,
    "GEL": 34.0,
    "KGS": 1.05,
    "AZN": 54.0,
}

print("ROOT_DIR:", ROOT_DIR)
print("INPUT_FILE:", INPUT_FILE)
print("OUT_TRAIN_READY:", OUT_TRAIN_READY)

ROOT_DIR: /home/kali/Documents/Diploma/hh-helper
INPUT_FILE: /home/kali/Documents/Diploma/hh-helper/data/vacancies_labeled_ml_dl.csv
OUT_TRAIN_READY: /home/kali/Documents/Diploma/hh-helper/artifacts/vacancies_train_ready.csv


In [13]:
# =========================
# 1) Competency taxonomy
# =========================
COMPETENCY_TAXONOMY = {
    "programming": {
        "python": ["python"],
        "sql": ["sql", "postgresql", "mysql", "clickhouse", "bigquery"],
        "scala": ["scala"],
        "java": ["java"],
        "c++": ["c++", "cpp"],
        "bash": ["bash", "shell scripting", "linux shell"],
    },
    "ml_dl_core": {
        "machine_learning": ["machine learning", "машинное обучение"],
        "deep_learning": [
            "deep learning",
            "deep neural networks",
            "neural networks",
            "нейронные сети",
            "глубокое обучение",
        ],
        "feature_engineering": ["feature engineering", "feature selection", "feature extraction"],
        "model_training": ["model training", "training models", "train models", "обучение моделей"],
        "model_evaluation": [
            "model evaluation",
            "evaluate models",
            "cross-validation",
            "offline evaluation",
            "валидация моделей",
        ],
        "hyperparameter_tuning": [
            "hyperparameter tuning",
            "parameter tuning",
            "grid search",
            "bayesian optimization",
            "optuna",
        ],
    },
    "llm_nlp": {
        "nlp": [
            "natural language processing",
            "обработка естественного языка",
            "text processing",
            "text mining",
            "nlp",
        ],
        "llm": [
            "large language model",
            "large language models",
            "foundation models",
            "language models",
            "llm",
            "gpt",
            "bert",
            "transformers",
            "transformers library",
        ],
        "rag": ["retrieval augmented generation", "retrieval-augmented generation", "rag"],
        "prompt_engineering": ["prompt engineering", "prompt design"],
        "fine_tuning": ["fine-tuning", "finetuning", "instruction tuning", "lora", "qlora"],
        "embeddings": ["embeddings", "sentence embeddings", "vector search", "semantic search"],
        "information_retrieval": [
            "information retrieval",
            "search relevance",
            "reranking",
            "ranking models",
            "retrieval systems",
        ],
        "text_generation": [
            "text generation",
            "summarization",
            "question answering",
            "chatbots",
            "conversational ai",
        ],
    },
    "cv_audio_multimodal": {
        "computer_vision": [
            "computer vision",
            "image processing",
            "object detection",
            "image segmentation",
            "segmentation models",
            "visual recognition",
            "компьютерное зрение",
        ],
        "ocr": ["optical character recognition", "ocr"],
        "video_ml": ["video analytics", "video processing", "video understanding"],
        "speech": [
            "speech recognition",
            "automatic speech recognition",
            "asr",
            "text to speech",
            "tts",
            "audio processing",
        ],
        "multimodal": ["multimodal", "vision-language", "vision language", "vlm"],
    },
    "ml_domains": {
        "recommender_systems": [
            "recommender systems",
            "recommendation systems",
            "recommendation engine",
            "personalized ranking",
            "рекомендательные системы",
        ],
        "time_series": ["time series", "forecasting", "demand forecasting", "временные ряды", "прогнозирование"],
        "anomaly_detection": ["anomaly detection", "outlier detection", "fraud detection"],
        "causal_inference": ["causal inference", "uplift modeling", "uplift modelling"],
        "experimentation": ["a/b testing", "ab testing", "online experiments", "experimentation"],
        "personalization": ["personalization", "personalisation"],
    },
    "frameworks_libraries": {
        "pytorch": ["pytorch", "torch"],
        "tensorflow": ["tensorflow"],
        "keras": ["keras"],
        "scikit_learn": ["scikit-learn", "sklearn"],
        "xgboost": ["xgboost"],
        "lightgbm": ["lightgbm", "light gbm"],
        "catboost": ["catboost"],
        "huggingface": ["huggingface", "hugging face"],
        "opencv": ["opencv"],
        "pandas": ["pandas"],
        "numpy": ["numpy"],
    },
    "data_engineering": {
        "spark": ["apache spark", "pyspark", "spark"],
        "airflow": ["apache airflow", "airflow"],
        "etl_elt": ["etl", "elt", "data pipelines", "pipeline development", "data pipeline"],
        "data_warehousing": ["data warehouse", "data warehousing", "snowflake", "redshift", "bigquery"],
        "streaming": ["kafka", "apache flink", "stream processing", "real-time data", "streaming data"],
        "feature_store": ["feature store", "feast"],
    },
    "mlops_production": {
        "mlops": ["mlops", "ml ops"],
        "model_deployment": [
            "model deployment",
            "deploy models",
            "productionize models",
            "production deployment",
            "deploying machine learning models",
        ],
        "model_serving": ["model serving", "inference service", "online inference", "real-time inference"],
        "inference_optimization": [
            "inference optimization",
            "latency optimization",
            "quantization",
            "distillation",
            "onnx",
            "tensorrt",
        ],
        "monitoring": ["model monitoring", "data drift", "concept drift", "monitoring ml systems"],
        "experiment_tracking": ["mlflow", "weights & biases", "wandb", "experiment tracking"],
        "ci_cd": ["ci/cd", "continuous integration", "continuous delivery", "github actions", "gitlab ci"],
        "docker": ["docker", "containerization", "контейнеризация"],
        "kubernetes": ["kubernetes", "k8s"],
        "api_backend": ["fastapi", "flask", "rest api", "grpc", "microservices", "backend services"],
    },
    "cloud_platform": {
        "aws": ["aws", "amazon web services", "sagemaker", "bedrock"],
        "gcp": ["gcp", "google cloud", "vertex ai", "google kubernetes engine"],
        "azure": ["azure", "azure ml", "microsoft azure"],
    },
    "math_stats": {
        "statistics": ["statistics", "statistical analysis", "математическая статистика", "статистика"],
        "probability": ["probability theory", "theory of probability", "теория вероятностей"],
        "linear_algebra": ["linear algebra", "линейная алгебра"],
    },
    "soft_and_process": {
        "communication": [
            "communication",
            "cross-functional",
            "stakeholder management",
            "коммуникация",
            "взаимодействие с командами",
        ],
        "leadership": [
            "leadership",
            "mentoring",
            "mentorship",
            "team lead",
            "technical leadership",
            "руководство",
            "менторство",
        ],
        "product_thinking": ["product thinking", "product mindset", "business impact", "product metrics"],
        "research": ["research", "state of the art", "sota", "research mindset"],
        "english": ["english", "upper-intermediate", "fluent english", "английский"],
        "agile": ["agile", "scrum", "kanban"],
    },
}

In [14]:
# =========================
# 2) Text normalization for keyword matching
# =========================
_RE_WS = re.compile(r"\s+")


def normalize_text(text: object) -> str:
    s = str(text).lower()

    replacements = {
        "k8s": "kubernetes",
        "pytorch lightning": "pytorch_lightning",
        "scikit learn": "scikit-learn",
        "hugging face": "huggingface",
        "a/b": "ab",
        "ci/cd": "ci_cd",
    }

    for old, new in replacements.items():
        s = s.replace(old, new)

    s = re.sub(r"[\/|]+", " ", s)
    s = re.sub(r"[\(\)\[\]\{\},;:!?]+", " ", s)
    s = _RE_WS.sub(" ", s)
    return s.strip()

In [15]:
# =========================
# 3) Safe regex search
# =========================

def pattern_to_regex(pattern: str) -> str:
    p = re.escape(pattern.strip().lower())
    return rf"(?<!\w){p}(?!\w)"


def skill_found(text: str, patterns: list[str]) -> bool:
    for pattern in patterns:
        if re.search(pattern_to_regex(pattern), text):
            return True
    return False

In [16]:
# =========================
# 4) Extract + flatten
# =========================

def extract_competencies(text: object, taxonomy: dict) -> dict:
    t = normalize_text(text)
    result: dict[str, list[str]] = {}

    for group_name, skills in taxonomy.items():
        found: list[str] = []
        for canonical_skill, patterns in skills.items():
            if skill_found(t, patterns):
                found.append(canonical_skill)
        result[group_name] = sorted(found)

    return result


def flatten_competencies(extracted: dict) -> list[str]:
    out: list[str] = []
    for _, skills in extracted.items():
        out.extend(skills)
    return sorted(set(out))


def join_items(items: list[str]) -> str:
    return ", ".join(items) if items else ""

In [17]:
# =========================
# 5) Aggregates
# =========================
AGGREGATES = {
    "is_llm_role": ["llm", "rag", "fine_tuning", "embeddings", "prompt_engineering"],
    "is_nlp_role": ["nlp", "information_retrieval", "text_generation", "llm"],
    "is_cv_role": ["computer_vision", "ocr", "video_ml", "multimodal"],
    "is_mlops_role": ["mlops", "model_deployment", "model_serving", "monitoring", "experiment_tracking"],
    "is_data_heavy_role": ["spark", "etl_elt", "streaming", "data_warehousing", "feature_store"],
    "is_researchy_role": ["research", "deep_learning", "hyperparameter_tuning", "model_training"],
}

In [18]:
# =========================
# 6) Load data
# =========================
if not INPUT_FILE.exists():
    raise FileNotFoundError(f"Input file not found: {INPUT_FILE}")

df = pd.read_csv(INPUT_FILE)

text_col = next((c for c in TEXT_COLUMNS_CANDIDATES if c in df.columns), None)
if text_col is None:
    raise ValueError(f"No text column found. Available columns: {list(df.columns)}")

# optional filter
if "ml_dl_label" in df.columns:
    df = df[df["ml_dl_label"] != "точно нет"].copy()

print("Text column:", text_col)
print("Rows:", len(df))

Text column: vacancy_text
Rows: 2307


In [19]:
# =========================
# 7) Extract competencies
# =========================
extracted = df[text_col].apply(lambda x: extract_competencies(x, COMPETENCY_TAXONOMY))

for group_name in COMPETENCY_TAXONOMY.keys():
    df[group_name] = extracted.apply(lambda x: join_items(x[group_name]))

df["all_competencies"] = extracted.apply(lambda x: join_items(flatten_competencies(x)))

# One-hot
for group_name, skills in COMPETENCY_TAXONOMY.items():
    for skill_name in skills.keys():
        df[f"skill__{skill_name}"] = extracted.apply(lambda x: int(skill_name in x[group_name]))

# Aggregates
for agg_name, agg_skills in AGGREGATES.items():
    df[agg_name] = extracted.apply(lambda x: int(any(s in flatten_competencies(x) for s in agg_skills)))

# Drop obvious label columns if present (optional)
df = df.drop(columns=[c for c in ["vacancy_text", "ml_dl_label", "ml_dl_binary", "confidence", "reason"] if c in df.columns], errors="ignore")

print("Done. Shape:", df.shape)

Done. Shape: (2307, 93)


In [20]:
# (Removed) Save full dataset
print("Skipping OUT_WITH_COMPETENCIES save: the notebook outputs only one train-ready file.")

Skipping OUT_WITH_COMPETENCIES save: the notebook outputs only one train-ready file.


In [21]:
# =========================
# 9) Keep only top-N competencies
# =========================
counter = Counter()
for value in df["all_competencies"].fillna(""):
    counter.update([x.strip() for x in str(value).split(",") if x.strip()])

top_skills = {skill for skill, _ in counter.most_common(TOP_N)}


def keep_top_skills(value: object) -> str:
    items = [x.strip() for x in str(value).split(",") if x.strip()]
    return ", ".join([x for x in items if x in top_skills])


df_top = df.copy()
df_top["all_competencies"] = df_top["all_competencies"].apply(keep_top_skills)
df_top = df_top[df_top["all_competencies"].str.strip().ne("")].copy()

# Rebuild one-hot for top skills only
skill_cols = [c for c in df_top.columns if c.startswith("skill__")]
df_top = df_top.drop(columns=skill_cols, errors="ignore")

for skill in sorted(top_skills):
    df_top[f"skill__{skill}"] = df_top["all_competencies"].apply(
        lambda x: int(skill in [s.strip() for s in str(x).split(",") if s.strip()])
    )

print("Top dataset shape:", df_top.shape)

Top dataset shape: (2005, 53)


In [22]:
# (Removed) Save top-N dataset
print("Skipping OUT_TOP30 save: the notebook outputs only one train-ready file.")

Skipping OUT_TOP30 save: the notebook outputs only one train-ready file.


In [23]:
# =========================
# 11) Salary parsing + train-ready dataset
# =========================

def detect_currency(text: object) -> str | None:
    if pd.isna(text):
        return None
    s = str(text).lower()

    if any(x in s for x in ["руб", "rur", "rub", "₽"]):
        return "RUB"
    if any(x in s for x in ["usd", "$", "доллар"]):
        return "USD"
    if any(x in s for x in ["eur", "€", "евро"]):
        return "EUR"
    if any(x in s for x in ["kzt", "₸", "тенге"]):
        return "KZT"
    if any(x in s for x in ["byn", "бел. руб", "белорус"]):
        return "BYN"
    if any(x in s for x in ["uah", "грн"]):
        return "UAH"
    if any(x in s for x in ["uzs", "сум"]):
        return "UZS"
    if any(x in s for x in ["gel", "лари"]):
        return "GEL"
    if any(x in s for x in ["kgs", "сом"]):
        return "KGS"
    if any(x in s for x in ["azn", "манат"]):
        return "AZN"

    return None


def parse_salary_to_rub(raw_value: object) -> float:
    if pd.isna(raw_value):
        return float("nan")

    s = str(raw_value).strip().lower()
    if s in {"", "none", "null", "nan", "не указана"}:
        return float("nan")

    currency = detect_currency(s)

    # remove separators inside numbers: "150 000" -> "150000"
    s = re.sub(r"(?<=\d)\s+(?=\d)", "", s)

    numbers = re.findall(r"\d+(?:[.,]\d+)?", s)
    values = [float(x.replace(",", ".")) for x in numbers]

    if not values:
        return float("nan")

    salary_value = values[0] if len(values) == 1 else (min(values[0], values[1]) + max(values[0], values[1])) / 2
    rate = CURRENCY_RATES_TO_RUB.get(currency or "", float("nan"))

    return salary_value * rate


if "salary" not in df_top.columns:
    raise KeyError("Column `salary` is required for salary parsing (not found in df_top).")

train_df = df_top.copy()
train_df["salary_rub"] = train_df["salary"].apply(parse_salary_to_rub)
train_df = train_df[train_df["salary_rub"].notna()].copy()
train_df = train_df[train_df["salary_rub"] > 0].copy()
train_df["salary_rub_log"] = np.log1p(train_df["salary_rub"])

# Region dummies
if "region" in train_df.columns:
    train_df["region"] = (
        train_df["region"].astype(str).str.strip().str.lower().replace({"nan": "unknown", "none": "unknown", "": "unknown"})
    )
    region_dummies = pd.get_dummies(train_df["region"], prefix="region", dtype="int8")
    train_df = pd.concat([train_df, region_dummies], axis=1)

# year_published
if "year_published" in train_df.columns:
    train_df["year_published"] = pd.to_numeric(train_df["year_published"], errors="coerce")
    train_df = train_df[train_df["year_published"].notna()].copy()
    train_df["year_published"] = train_df["year_published"].astype("int16")

# Convert binary cols
binary_cols = [c for c in train_df.columns if c.startswith("skill__") or c.startswith("is_") or c.startswith("region_")]
for c in binary_cols:
    train_df[c] = pd.to_numeric(train_df[c], errors="coerce").fillna(0).astype("int8")

# Drop text-ish columns
train_df = train_df.drop(columns=[c for c in ["salary", "region", "all_competencies"] if c in train_df.columns], errors="ignore")

train_df = train_df.drop_duplicates().copy()
print("Train-ready shape:", train_df.shape)

Train-ready shape: (483, 116)


In [24]:
# =========================
# 12) Save train-ready dataset
# =========================
train_df.to_csv(OUT_TRAIN_READY, index=False)

# Basic sanity output
target_col = "salary_rub_log"
feature_cols = [c for c in train_df.columns if c != target_col]
print("Saved:", OUT_TRAIN_READY)
print("Shape:", train_df.shape)
print("Target:", target_col)
print("Num features:", len(feature_cols))

Saved: /home/kali/Documents/Diploma/hh-helper/artifacts/vacancies_train_ready.csv
Shape: (483, 116)
Target: salary_rub_log
Num features: 115
